In [0]:
# Display all columns and attributes from Silver tables

silver_schema = "big_data.silver"

print("=" * 100)
print("SILVER LAYER - TABLE SCHEMAS")
print("=" * 100)

# List of silver tables to inspect
tables = ['orders', 'products_enriched', 'order_products']

for table_name in tables:
    full_table_name = f"{silver_schema}.{table_name}"
    print(f"\n{'='*100}")
    print(f"Table: {full_table_name}")
    print("=" * 100)
    
    # Get table schema using DESCRIBE
    schema_df = spark.sql(f"DESCRIBE {full_table_name}")
    display(schema_df)
    
    # Get row count
    row_count = spark.table(full_table_name).count()
    print(f"\nTotal rows: {row_count:,}")
    
    # Show sample data
    print(f"\nSample data (first 3 rows):")
    display(spark.table(full_table_name).limit(3))
    print("\n")

Quantos dias em media um usuario tem entre uma compra e outra; Periodo do dia com maior venda; Categoria de produtos que possuem mais re-order 

 

In [0]:
from pyspark.sql.functions import col, count, countDistinct, avg, sum as _sum, round as _round, max as _max, min as _min


In [0]:

# Load silver tables
orders = spark.table("big_data.silver.orders")
order_products = spark.table("big_data.silver.order_products")

print("="*80)
print("SALES ANALYSIS BY TIME PERIOD")
print("="*80)

# Analysis 1: Sales by Period of Day
print("\nSales by Period of Day (morning/afternoon/evening/night):\n")

sales_by_period = orders.join(order_products, "order_id") \
    .groupBy("period_of_day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),  # CORREÇÃO AQUI
        count("product_id").alias("total_items_sold")
    )

# Total correto de pedidos (sem duplicação)
total_orders = orders.select("order_id").distinct().count()

# Adicionando coluna de porcentagem
sales_by_period = sales_by_period.withColumn(
    "percentage_orders",
    _round((col("total_orders") / total_orders) * 100, 2)
).orderBy(col("total_orders").desc())

display(sales_by_period)

print(f"\nTotal Orders: {total_orders:,}\n")

# Analysis 2: Sales by Hour of Day
print("\n" + "="*80)
print("Sales by Hour of Day (0-23):")
print("="*80 + "\n")

sales_by_hour = orders.join(order_products, "order_id") \
    .groupBy("order_hour_of_day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),  # CORREÇÃO AQUI
        count("product_id").alias("total_items_sold")
    )

# Adicionando porcentagem também aqui
sales_by_hour = sales_by_hour.withColumn(
    "percentage_orders",
    _round((col("total_orders") / total_orders) * 100, 2)
).orderBy("order_hour_of_day")

display(sales_by_hour)

# Find busiest hours
print("\nTop 5 Busiest Hours:\n")

top_hours = sales_by_hour.orderBy(col("total_orders").desc()).limit(5).collect()

for i, row in enumerate(top_hours, 1):
    print(f"{i}. Hour {row['order_hour_of_day']}:00 - {row['total_orders']:,} orders ({row['percentage_orders']}%) - {row['total_items_sold']:,} items")

In [0]:

# Load silver tables
order_products = spark.table("big_data.silver.order_products")
products_enriched = spark.table("big_data.silver.products_enriched")

print("="*80)
print("REORDER ANALYSIS BY PRODUCT CATEGORY")
print("="*80)

# Join order_products with products to get category information
orders_with_categories = order_products.join(products_enriched, "product_id")

# Analysis 1: Reorders by Department
print("\nReorders by Department:\n")

reorders_by_dept = orders_with_categories.groupBy("department") \
    .agg(
        count("*").alias("total_items"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
    ) \
    .withColumn("reorder_rate", _round((col("reordered_items") / col("total_items")) * 100, 2)) \
    .orderBy(col("reordered_items").desc())

display(reorders_by_dept)

print("\nTop 5 Departments by Reorder Count:\n")
top_depts = reorders_by_dept.limit(5).collect()
for i, row in enumerate(top_depts, 1):
    print(f"  {i}. {row['department']}: {row['reordered_items']:,} reorders ({row['reorder_rate']}% reorder rate)")

# Analysis 2: Reorders by Aisle
print("\n" + "="*80)
print("Reorders by Aisle (Top 10):\n")

reorders_by_aisle = orders_with_categories.groupBy("aisle") \
    .agg(
        count("*").alias("total_items"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
    ) \
    .withColumn("reorder_rate", _round((col("reordered_items") / col("total_items")) * 100, 2)) \
    .orderBy(col("reordered_items").desc()) \
    .limit(10)

display(reorders_by_aisle)

print("\nTop 5 Aisles by Reorder Count:\n")
top_aisles = reorders_by_aisle.limit(5).collect()
for i, row in enumerate(top_aisles, 1):
    print(f"  {i}. {row['aisle']}: {row['reordered_items']:,} reorders ({row['reorder_rate']}% reorder rate)")

In [0]:
from pyspark.sql.functions import col, avg, count, min as _min, max as _max, round as _round, percentile_approx

# Load silver orders table
orders = spark.table("big_data.silver.orders")

print("="*80)
print("AVERAGE DAYS BETWEEN ORDERS ANALYSIS")
print("="*80)

# Overall average days between orders (excluding first orders)
print("\nOverall Statistics:\n")

overall_stats = orders.filter(col("is_first_order") == False) \
    .agg(
        _round(avg("days_since_prior_order"), 2).alias("avg_days"),
        _round(_min("days_since_prior_order"), 2).alias("min_days"),
        _round(_max("days_since_prior_order"), 2).alias("max_days"),
        _round(percentile_approx("days_since_prior_order", 0.5), 2).alias("median_days"),
        count("*").alias("total_repeat_orders")
    )

display(overall_stats)

stats = overall_stats.collect()[0]
print(f"\nOverall Average: {stats['avg_days']} days between orders")
print(f"Median: {stats['median_days']} days")
print(f"Range: {stats['min_days']} to {stats['max_days']} days")
print(f"Total Repeat Orders: {stats['total_repeat_orders']:,}")

# Average days between orders per user
print("\n" + "="*80)
print("👥 Average Days Between Orders by User (Sample of 10 users):\n")

user_avg_days = orders.filter(col("is_first_order") == False) \
    .groupBy("user_id") \
    .agg(
        _round(avg("days_since_prior_order"), 2).alias("avg_days_between_orders"),
        count("*").alias("repeat_order_count")
    ) \
    .orderBy(col("repeat_order_count").desc())

display(user_avg_days.limit(10))

# Distribution of average days between orders
print("\n" + "="*80)
print("Distribution of Days Between Orders:\n")

days_distribution = orders.filter(col("is_first_order") == False) \
    .selectExpr(
        "CASE \
            WHEN days_since_prior_order <= 7 THEN '1-7 days' \
            WHEN days_since_prior_order <= 14 THEN '8-14 days' \
            WHEN days_since_prior_order <= 21 THEN '15-21 days' \
            WHEN days_since_prior_order <= 30 THEN '22-30 days' \
            ELSE '30+ days' \
        END as days_range"
    ) \
    .groupBy("days_range") \
    .agg(count("*").alias("order_count")) \
    .orderBy(
        col("days_range").isin('1-7 days', '8-14 days', '15-21 days', '22-30 days', '30+ days').desc(),
        "days_range"
    )

display(days_distribution)

print("\nCustomer Purchase Frequency Insights:\n")
dist_data = days_distribution.collect()
for row in dist_data:
    print(f"  • {row['days_range']}: {row['order_count']:,} orders")

In [0]:
# Load silver tables
orders = spark.table("big_data.silver.orders")
order_products = spark.table("big_data.silver.order_products")

print("="*80)
print("CUSTOMER RELATIONSHIP ANALYSIS")
print("="*80)

# Analysis 1: Customer Order Frequency
print("\nCustomer Order Frequency Distribution:\n")

customer_orders = orders.groupBy("user_id") \
    .agg(
        count("order_id").alias("total_orders"),
        _max("order_number").alias("max_order_number")
    )

# Distribution of customers by order count
order_frequency_dist = customer_orders.selectExpr(
    "CASE \
        WHEN total_orders = 1 THEN '1 order (New)' \
        WHEN total_orders BETWEEN 2 AND 5 THEN '2-5 orders (Occasional)' \
        WHEN total_orders BETWEEN 6 AND 10 THEN '6-10 orders (Regular)' \
        WHEN total_orders BETWEEN 11 AND 20 THEN '11-20 orders (Frequent)' \
        ELSE '20+ orders (Loyal)' \
    END as customer_segment"
) \
.groupBy("customer_segment") \
.agg(count("*").alias("customer_count")) \
.orderBy(
    col("customer_segment").isin('1 order (New)', '2-5 orders (Occasional)', '6-10 orders (Regular)', '11-20 orders (Frequent)', '20+ orders (Loyal)').desc(),
    "customer_segment"
)

display(order_frequency_dist)

total_customers = customer_orders.count()
print(f"\nTotal Customers: {total_customers:,}\n")

seg_data = order_frequency_dist.collect()
for row in seg_data:
    percentage = (row['customer_count'] / total_customers) * 100
    print(f"  • {row['customer_segment']}: {row['customer_count']:,} customers ({percentage:.1f}%)")

# Analysis 2: Top Customers (Most Loyal)
print("\n" + "="*80)
print("Top 10 Most Loyal Customers:\n")

# Join with order_products to get total items purchased
top_customers = orders.join(order_products, "order_id") \
    .groupBy("user_id") \
    .agg(
        count("order_id").alias("total_orders"),
        count("product_id").alias("total_items_purchased"),
        _round(avg("days_since_prior_order"), 2).alias("avg_days_between_orders")
    ) \
    .orderBy(col("total_orders").desc()) \
    .limit(10)

display(top_customers)

# Analysis 3: Customer Lifecycle Metrics
print("\n" + "="*80)
print("Customer Lifecycle Metrics:\n")

lifecycle_metrics = orders.agg(
    count("user_id").alias("total_orders"),
    _round(avg("order_number"), 2).alias("avg_orders_per_customer"),
    _sum(when(col("is_first_order") == True, 1).otherwise(0)).alias("first_time_orders"),
    _sum(when(col("is_first_order") == False, 1).otherwise(0)).alias("repeat_orders")
) \
.withColumn("repeat_order_rate", _round((col("repeat_orders") / col("total_orders")) * 100, 2))

display(lifecycle_metrics)

metrics = lifecycle_metrics.collect()[0]
print(f"\nKey Metrics:")
print(f"  • Average Orders per Customer: {metrics['avg_orders_per_customer']}")
print(f"  • First-Time Orders: {metrics['first_time_orders']:,}")
print(f"  • Repeat Orders: {metrics['repeat_orders']:,}")
print(f"  • Repeat Order Rate: {metrics['repeat_order_rate']}%")

# Analysis 4: Customer Engagement by Order Size
print("\n" + "="*80)
print("Average Basket Size by Customer Segment:\n")

customer_basket = orders.join(order_products, "order_id") \
    .groupBy("user_id", "order_id") \
    .agg(count("product_id").alias("items_in_basket"))

avg_basket_by_orders = customer_orders.join(customer_basket, "user_id") \
    .selectExpr(
        "CASE \
            WHEN total_orders = 1 THEN 'New (1 order)' \
            WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional (2-5)' \
            WHEN total_orders BETWEEN 6 AND 10 THEN 'Regular (6-10)' \
            WHEN total_orders BETWEEN 11 AND 20 THEN 'Frequent (11-20)' \
            ELSE 'Loyal (20+)' \
        END as segment",
        "items_in_basket"
    ) \
    .groupBy("segment") \
    .agg(
        _round(avg("items_in_basket"), 2).alias("avg_basket_size")
    ) \
    .orderBy(
        col("segment").isin('New (1 order)', 'Occasional (2-5)', 'Regular (6-10)', 'Frequent (11-20)', 'Loyal (20+)').desc(),
        "segment"
    )

display(avg_basket_by_orders)